In [ ]:
import torch
import gpytorch
import numpy as np
import pandas as pd
import time
import warnings
from pathlib import Path
from typing import List, Tuple, Dict

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import (
    accuracy_score, roc_auc_score, average_precision_score,
    brier_score_loss, confusion_matrix, log_loss
)
from sklearn.calibration import calibration_curve
from tqdm import tqdm

warnings.filterwarnings("ignore")


# ============================================================================
# GP MODEL CLASSES
# ============================================================================

class GPRegressionModel_VariationalSparse(gpytorch.models.ApproximateGP):
    """Variational Sparse Gaussian Process Model"""
    
    def __init__(self, inducing_points):
        variational_distribution = gpytorch.variational.CholeskyVariationalDistribution(
            inducing_points.size(0)
        )
        variational_strategy = gpytorch.variational.VariationalStrategy(
            self, inducing_points, variational_distribution, learn_inducing_locations=True
        )
        super(GPRegressionModel_VariationalSparse, self).__init__(variational_strategy)
        
        self.mean_module = gpytorch.means.ConstantMean()
        self.covar_module = gpytorch.kernels.ScaleKernel(gpytorch.kernels.RBFKernel())
        
        # Parameters for MacKay's approximation
        self.a = torch.nn.Parameter(torch.tensor(1.0))  # mean scaler
        self.b = torch.nn.Parameter(torch.tensor(0.0))  # bias
        self.c = torch.nn.Parameter(torch.tensor(torch.pi/8))  # variance scaler

    def forward(self, x):
        mean = self.mean_module(x)
        covariance = self.covar_module(x)
        return gpytorch.distributions.MultivariateNormal(mean, covariance)
    
    def predict_probability(self, x, likelihood):
        """MacKay's approximation for classification probabilities"""
        self.eval()
        likelihood.eval()
        
        with torch.no_grad(), gpytorch.settings.fast_pred_var():
            pred = likelihood(self(x))
            pred_mean = pred.mean
            pred_variance = pred.variance
            
            # Prevent divisions by zero
            c_clamped = torch.clamp(self.c, min=1e-6, max=20.0)
            
            # MacKay's approximation
            numerator = pred_mean * self.a + self.b
            denominator = torch.sqrt(1 + (torch.pi * pred_variance * c_clamped) / 8)
            prob = torch.sigmoid(numerator / denominator)
        
        return prob


class EarlyStopping:
    """Early stopping utility"""
    
    def __init__(self, patience=15, min_delta=1e-4):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_loss = None
        
    def check_early_stopping(self, epoch, loss):
        if self.best_loss is None:
            self.best_loss = loss
            return False
        
        if loss < self.best_loss - self.min_delta:
            self.best_loss = loss
            self.counter = 0
        else:
            self.counter += 1
            
        return self.counter >= self.patience
    
    def reset(self):
        self.counter = 0
        self.best_loss = None


# ============================================================================
# HEART DISEASE DETECTION CLASS
# ============================================================================

class HeartDiseaseVSGP:
    """
    Heart Disease Detection using Variational Sparse Gaussian Process
    with MacKay's approximation for binary classification
    """
    
    def __init__(self, data_path: str, n_folds: int = 5, 
                 num_inducing_points: int = 50,
                 use_mackay: bool = True,
                 batch_size: int = 200):
        
        self.data_path = data_path
        self.n_folds = n_folds
        self.num_inducing_points = num_inducing_points
        self.use_mackay = use_mackay
        self.batch_size = batch_size
        
        # Training parameters
        self.learning_rate = 0.05
        self.max_epochs = 1000
        self.random_state = 42
        
        # Column mapping for Heart Disease dataset
        self.column_names = [
            'age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg',
            'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal', 'target'
        ]
        
        # Device setup
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        print(f"Using device: {self.device}")
        
        # Results storage
        self.results = []
        
        np.random.seed(self.random_state)
        torch.manual_seed(self.random_state)
    
    def load_and_preprocess_data(self):
        """Load and preprocess Heart Disease data"""
        print("\n" + "="*60)
        print("LOADING AND PREPROCESSING HEART DISEASE DATA")
        print("="*60)
        
        df = pd.read_csv(self.data_path)
        print(f"Original shape: {df.shape}")
        
        # Check if columns need to be named
        if not all(col in df.columns for col in ['age', 'target']):
            if len(df.columns) == 14:
                df.columns = self.column_names
                print("Assigned column names to dataset")
        
        # Handle missing values (marked as '?' in some heart disease datasets)
        df = df.replace('?', np.nan)
        
        # Convert all columns to numeric
        for col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
        
        # Fill missing values with median
        df = df.fillna(df.median())
        
        # Separate features and target
        X = df.drop('target', axis=1)
        y = df['target']
        
        # Convert target to binary (0 = no disease, 1 = disease)
        # Original dataset: 0 = no disease, 1-4 = disease levels
        y = (y > 0).astype(int)
        
        print(f"Features shape: {X.shape}")
        print(f"Feature names: {list(X.columns)}")
        print(f"\nTarget distribution:")
        print(f"  No disease (0): {(y == 0).sum()} ({(y == 0).mean():.2%})")
        print(f"  Disease (1):    {(y == 1).sum()} ({(y == 1).mean():.2%})")
        
        return X.values, y.values
    
    def expected_calibration_error(self, y_true, y_prob, n_bins=10):
        """Calculate Expected Calibration Error"""
        try:
            prob_true, prob_pred = calibration_curve(
                y_true, y_prob, n_bins=n_bins, strategy='uniform'
            )
            bins = np.linspace(0, 1, n_bins + 1)
            counts, _ = np.histogram(y_prob, bins=bins)
            N = len(y_true)
            weights = counts / max(N, 1)
            m = min(len(prob_true), len(weights))
            return float(np.sum(weights[:m] * np.abs(prob_true[:m] - prob_pred[:m])))
        except:
            return 0.0
    
    def calculate_kl_divergence(self, y_true, y_prob):
        """Calculate KL Divergence between true and predicted distributions"""
        epsilon = 1e-10
        y_prob = np.clip(y_prob, epsilon, 1 - epsilon)
        
        # True distribution
        p = np.array([1 - y_true.mean(), y_true.mean()])
        
        # Predicted distribution
        q_pos = y_prob[y_true == 1].mean() if (y_true == 1).any() else 0.5
        q_neg = y_prob[y_true == 0].mean() if (y_true == 0).any() else 0.5
        q = np.array([1 - (q_pos + q_neg)/2, (q_pos + q_neg)/2])
        
        kl_div = np.sum(p * np.log(p / (q + epsilon)))
        return kl_div
    
    def gaussian_pdf(self, x, mean, var):
        """Gaussian probability density function"""
        return (1 / np.sqrt(2 * np.pi * var)) * np.exp(-0.5 * ((x - mean) ** 2) / var)
    
    def train_model(self, X_train, y_train, fold_num):
        """Train Variational Sparse GP model"""
        print(f"\n[Fold {fold_num}] Training Variational Sparse GP...")
        print(f"Training samples: {len(X_train)}")
        print(f"Inducing points: {self.num_inducing_points}")
        print(f"USE_MACKAY_APPROXIMATION: {self.use_mackay}")
        
        train_start = time.time()
        
        # Convert to tensors
        X_train_tensor = torch.FloatTensor(X_train).to(self.device)
        y_train_tensor = torch.FloatTensor(y_train).to(self.device)
        
        # Initialize inducing points using K-means
        print(f"[Fold {fold_num}] Initializing inducing points with K-means...")
        n_clusters = min(self.num_inducing_points, len(X_train))
        kmeans = KMeans(n_clusters=n_clusters, 
                       init='k-means++', 
                       random_state=self.random_state,
                       n_init=10).fit(X_train)
        inducing_points = torch.FloatTensor(kmeans.cluster_centers_).to(self.device)
        
        # Initialize model and likelihood
        model = GPRegressionModel_VariationalSparse(inducing_points).to(self.device)
        likelihood = gpytorch.likelihoods.GaussianLikelihood().to(self.device)
        
        # PART 1: Train GP hyperparameters
        model.train()
        likelihood.train()
        
        # Optimizer includes both model and likelihood parameters
        parameters = [
            {'params': model.parameters()},
            {'params': likelihood.parameters()}
        ]
        optimizer1 = torch.optim.Adam(parameters, lr=self.learning_rate)
        
        # ELBO loss for variational inference
        mll = gpytorch.mlls.VariationalELBO(likelihood, model, num_data=len(X_train))
        
        es = EarlyStopping(patience=20, min_delta=1e-4)
        
        pbar = tqdm(range(self.max_epochs), desc=f"[Fold {fold_num}] GP Training", leave=False)
        for epoch in pbar:
            optimizer1.zero_grad()
            output = model(X_train_tensor)
            loss = -mll(output, y_train_tensor)
            loss.backward()
            optimizer1.step()
            
            pbar.set_postfix({'loss': f'{loss.item():.4f}'})
            
            if es.check_early_stopping(epoch, loss.item()):
                print(f"Early stopping at epoch {epoch}")
                break
        
        # Print learned parameters
        print(f"Learned parameters:")
        print(f"  Lengthscale: {model.covar_module.base_kernel.lengthscale.item():.4f}")
        print(f"  Outputscale: {model.covar_module.outputscale.item():.4f}")
        print(f"  Noise: {likelihood.noise.item():.4f}")
        
        # PART 2: Train MacKay's scaling factors (if enabled)
        if self.use_mackay:
            print(f"Training MacKay scaling factors...")
            
            # Freeze GP parameters
            for param in model.mean_module.parameters():
                param.requires_grad = False
            for param in model.covar_module.parameters():
                param.requires_grad = False
            for param in model.variational_strategy.parameters():
                param.requires_grad = False
            for param in likelihood.parameters():
                param.requires_grad = False
            
            # Ensure MacKay parameters require grad
            model.a.requires_grad = True
            model.b.requires_grad = True
            model.c.requires_grad = True
            
            optimizer2 = torch.optim.Adam([model.a, model.b, model.c], lr=self.learning_rate)
            bce = torch.nn.BCELoss()
            es.reset()
            
            pbar2 = tqdm(range(self.max_epochs), desc=f"[Fold {fold_num}] MacKay Training", leave=False)
            for epoch in pbar2:
                model.train()
                optimizer2.zero_grad()
                
                # Forward pass - compute predictions with gradients for a, b, c
                output = model(X_train_tensor)
                pred = likelihood(output)
                pred_mean = pred.mean
                pred_variance = pred.variance
                
                # MacKay's approximation with gradients
                c_clamped = torch.clamp(model.c, min=1e-6, max=20.0)
                numerator = pred_mean * model.a + model.b
                denominator = torch.sqrt(1 + (torch.pi * pred_variance * c_clamped) / 8)
                probs = torch.sigmoid(numerator / denominator)
                
                loss = bce(probs, y_train_tensor)
                loss.backward()
                optimizer2.step()
                
                pbar2.set_postfix({'loss': f'{loss.item():.4f}'})
                
                if es.check_early_stopping(epoch, loss.item()):
                    print(f"Early stopping at epoch {epoch}")
                    break
            
            print(f"Scaling factors - a: {model.a.item():.4f}, b: {model.b.item():.4f}, c: {model.c.item():.4f}")
        
        train_time = time.time() - train_start
        
        return model, likelihood, train_time
    
    def evaluate_model(self, model, likelihood, X_test, y_test, fold_num):
        """Evaluate model with batched predictions"""
        print(f"\n[Fold {fold_num}] Evaluating model...")
        
        test_start = time.time()
        model.eval()
        likelihood.eval()
        
        y_pred = []
        y_prob = []
        
        # Batch predictions for memory efficiency
        n_samples = len(X_test)
        
        for i in range(0, n_samples, self.batch_size):
            batch_end = min(i + self.batch_size, n_samples)
            X_batch = torch.FloatTensor(X_test[i:batch_end]).to(self.device)
            
            if self.use_mackay:
                # MacKay's approximation
                with torch.no_grad():
                    probs = model.predict_probability(X_batch, likelihood)
                    probs = probs.cpu().numpy()
            else:
                # Normalized likelihood ratio
                with torch.no_grad(), gpytorch.settings.fast_pred_var():
                    predictions = likelihood(model(X_batch))
                    means = predictions.mean.cpu().numpy()
                    vars = predictions.variance.cpu().numpy()
                    
                    pdf_1 = self.gaussian_pdf(1, means, vars)
                    pdf_0 = self.gaussian_pdf(0, means, vars)
                    probs = pdf_1 / (pdf_1 + pdf_0 + 1e-10)
            
            y_prob.extend(probs)
            y_pred.extend((probs > 0.5).astype(int))
            
            # Cleanup
            del X_batch
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        
        test_time = time.time() - test_start
        
        y_prob = np.array(y_prob)
        y_pred = np.array(y_pred)
        
        # Calculate comprehensive metrics
        accuracy = accuracy_score(y_test, y_pred)
        
        # Handle cases where only one class is present
        try:
            roc_auc = roc_auc_score(y_test, y_prob)
        except:
            roc_auc = np.nan
        
        try:
            pr_auc = average_precision_score(y_test, y_prob)
        except:
            pr_auc = np.nan
        
        brier = brier_score_loss(y_test, y_prob)
        ece = self.expected_calibration_error(y_test, y_prob)
        kl_div = self.calculate_kl_divergence(y_test, y_prob)
        
        # Cross-entropy loss
        try:
            ce_loss = log_loss(y_test, y_prob)
        except:
            ce_loss = np.nan
        
        return {
            'accuracy': accuracy,
            'roc_auc': roc_auc,
            'pr_auc': pr_auc,
            'brier_score': brier,
            'ece': ece,
            'kl_divergence': kl_div,
            'cross_entropy': ce_loss,
            'test_time': test_time,
            'test_time_per_sample': test_time / len(X_test)
        }
    
    def run_cross_validation(self):
        """Run stratified k-fold cross-validation"""
        print("\n" + "="*60)
        print(f"VARIATIONAL SPARSE GP - {self.n_folds}-FOLD CV")
        print("="*60)
        print(f"Inducing points: {self.num_inducing_points}")
        print(f"MacKay approximation: {self.use_mackay}")
        print(f"Batch size: {self.batch_size}")
        
        # Load data
        X, y = self.load_and_preprocess_data()
        
        # Setup cross-validation
        skf = StratifiedKFold(
            n_splits=self.n_folds,
            shuffle=True,
            random_state=self.random_state
        )
        
        all_metrics = []
        oof_probs = np.zeros(len(X), dtype=float)
        oof_true = y.copy()
        
        for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), 1):
            print(f"\n{'='*60}")
            print(f"FOLD {fold}/{self.n_folds}")
            print(f"{'='*60}")
            
            # Split data
            X_train, X_test = X[train_idx], X[test_idx]
            y_train, y_test = y[train_idx], y[test_idx]
            
            # Standardize
            scaler = StandardScaler()
            X_train = scaler.fit_transform(X_train)
            X_test = scaler.transform(X_test)
            
            print(f"Train size: {len(X_train)}, Test size: {len(X_test)}")
            print(f"Train disease rate: {y_train.mean():.2%}")
            print(f"Test disease rate: {y_test.mean():.2%}")
            
            # Train model
            model, likelihood, train_time = self.train_model(X_train, y_train, fold)
            
            # Evaluate model
            metrics = self.evaluate_model(model, likelihood, X_test, y_test, fold)
            
            # Store OOF predictions
            X_test_tensor = torch.FloatTensor(X_test).to(self.device)
            model.eval()
            likelihood.eval()
            
            with torch.no_grad():
                if self.use_mackay:
                    oof_pred = model.predict_probability(X_test_tensor, likelihood).cpu().numpy()
                else:
                    with gpytorch.settings.fast_pred_var():
                        pred = likelihood(model(X_test_tensor))
                        means = pred.mean.cpu().numpy()
                        vars = pred.variance.cpu().numpy()
                        pdf_1 = self.gaussian_pdf(1, means, vars)
                        pdf_0 = self.gaussian_pdf(0, means, vars)
                        oof_pred = pdf_1 / (pdf_1 + pdf_0 + 1e-10)
            
            oof_probs[test_idx] = oof_pred
            
            # Add fold info to metrics
            metrics['fold'] = fold
            metrics['train_samples'] = len(X_train)
            metrics['test_samples'] = len(X_test)
            metrics['train_time'] = train_time
            
            all_metrics.append(metrics)
            
            # Print fold results
            print(f"\n[Fold {fold}] Results:")
            print(f"  Accuracy:          {metrics['accuracy']:.4f}")
            print(f"  ROC-AUC:           {metrics['roc_auc']:.4f}")
            print(f"  PR-AUC:            {metrics['pr_auc']:.4f}")
            print(f"  Brier Score:       {metrics['brier_score']:.4f}")
            print(f"  ECE:               {metrics['ece']:.4f}")
            print(f"  KL Divergence:     {metrics['kl_divergence']:.4f}")
            print(f"  Cross-Entropy:     {metrics['cross_entropy']:.4f}")
            print(f"  Train Time:        {train_time:.2f}s")
            print(f"  Test Time:         {metrics['test_time']:.2f}s")
            
            # Cleanup
            del model, likelihood, X_train, X_test
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        
        # Print summary
        self.print_summary(all_metrics, oof_true, oof_probs)
        
        return all_metrics, oof_true, oof_probs
    
    def print_summary(self, all_metrics: List[Dict], y_true, y_prob):
        """Print comprehensive summary statistics"""
        print("\n" + "="*60)
        print("CROSS-VALIDATION SUMMARY")
        print("="*60)
        
        df_metrics = pd.DataFrame(all_metrics)
        
        # Per-fold table
        print("\nPer-Fold Results:")
        display_cols = ['fold', 'accuracy', 'roc_auc', 'pr_auc', 'brier_score',
                       'ece', 'kl_divergence', 'cross_entropy', 'train_time', 'test_time']
        print(df_metrics[display_cols].to_string(index=False, float_format='%.4f'))
        
        # Mean ± std
        print("\n" + "="*60)
        print("Mean ± Std Across Folds:")
        print("="*60)
        
        metrics_to_summarize = [
            'accuracy', 'roc_auc', 'pr_auc', 'brier_score',
            'ece', 'kl_divergence', 'cross_entropy',
            'train_time', 'test_time', 'test_time_per_sample'
        ]
        
        for metric in metrics_to_summarize:
            if metric in df_metrics.columns:
                values = df_metrics[metric].dropna().values
                if len(values) > 0:
                    mean_val = np.mean(values)
                    std_val = np.std(values)
                    print(f"{metric.replace('_', ' ').title():30s} {mean_val:.4f} ± {std_val:.4f}")
        
        # Overall OOF performance
        print("\n" + "="*60)
        print("Overall Out-of-Fold Performance:")
        print("="*60)
        
        oof_accuracy = accuracy_score(y_true, (y_prob >= 0.5).astype(int))
        oof_roc_auc = roc_auc_score(y_true, y_prob)
        oof_brier = brier_score_loss(y_true, y_prob)
        oof_ece = self.expected_calibration_error(y_true, y_prob)
        oof_kl = self.calculate_kl_divergence(y_true, y_prob)
        
        print(f"Accuracy:          {oof_accuracy:.4f}")
        print(f"ROC-AUC:           {oof_roc_auc:.4f}")
        print(f"Brier Score:       {oof_brier:.4f}")
        print(f"ECE:               {oof_ece:.4f}")
        print(f"KL Divergence:     {oof_kl:.4f}")


# ============================================================================
# MAIN EXECUTION
# ============================================================================

if __name__ == "__main__":
    # Configuration for Heart Disease Dataset
    DATA_PATH = "../dataset/heart.csv"  # Update with your file path
    N_FOLDS = 5
    NUM_INDUCING_POINTS = 50  # Smaller dataset, fewer inducing points needed
    USE_MACKAY = True
    BATCH_SIZE = 200  # Smaller batches for smaller dataset
    
    print("="*60)
    print("HEART DISEASE DETECTION - VARIATIONAL SPARSE GP")
    print("="*60)
    print(f"Configuration:")
    print(f"  Data: {DATA_PATH}")
    print(f"  Folds: {N_FOLDS}")
    print(f"  Inducing points: {NUM_INDUCING_POINTS}")
    print(f"  MacKay approximation: {USE_MACKAY}")
    print(f"  Batch size: {BATCH_SIZE}")
    print("="*60)
    
    # Run detection
    detector = HeartDiseaseVSGP(
        data_path=DATA_PATH,
        n_folds=N_FOLDS,
        num_inducing_points=NUM_INDUCING_POINTS,
        use_mackay=USE_MACKAY,
        batch_size=BATCH_SIZE
    )
    
    metrics, y_true, y_prob = detector.run_cross_validation()
    
    print("\n" + "="*60)
    print("CROSS-VALIDATION COMPLETE!")
    print("="*60)